<a href="https://colab.research.google.com/github/ketchicken/cse144-spring-2026-final-project/blob/main/cse144_final_project_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()
# KGAT_6a0559d6033445e30d0c9f0daefc641b

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

ucsc_cse_144_spring_2026_final_project_path = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')

print('Data source import complete.')


100%|██████████| 120M/120M [00:01<00:00, 104MB/s] 

Extracting files...


Data source import complete.


In [4]:
# Imports and Setup
import os, random
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, models, io
from PIL import Image
from torchvision.transforms import v2 as tfv2
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.nn.modules import pooling
from torch.nn.modules.pooling import MaxUnpool2d
import torch.optim as optim
import csv
import copy

device =  torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    ''' For reproducible results across runs '''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

In [7]:
# Functions called for Training

def accuracy(loader, model):
    model.eval()
    correct = 0
    total = 0
    for data, label in loader:
        data, label = data.to(device), label.to(device)
        output = model(data)
        prediction=output.argmax(1)
        total += label.size(0)
        correct += prediction.eq(label.view_as(prediction)).sum().item()

    return 100 * correct / total

def run_one_epoch(loader, model, epoch):
    model.train()
    total_loss = 0.0

    for data, label in loader:
        # Applying Cutmix/Mixup 50% of the time
        if torch.rand(1).item() > 0.7:
            data, label = cutmix_or_mixup(data, label)

        # Train loop
        data, label = data.to(device), label.to(device)
        optimizer.zero_grad() # reset optimizer gradients
        output = model(data)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    acc = accuracy(loader, model)
    return total_loss / len(loader), acc


In [6]:
# Validation Functions

def validate(loader, model):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for data, label in loader:
            data, label = data.to(device), label.to(device)
            output = model(data)
            loss = criterion(output, label)
            total_loss += loss.item()

    acc = accuracy(loader, model)
    return total_loss / len(loader), acc

    def generate_labels(loader, model):
        model.eval()
        labels = []
        with torch.no_grad():
            for data, _ in loader:
                data = data.to(device)
                output = model(data)
                prediction = torch.argmax(output, dim=1)
                labels.append(prediction.cpu())

        return torch.cat(labels)


In [9]:
# Training and Validating
def one_fold(fold, model, num_epochs, ckpt_path, csv_path, train_data, train_loader, val_loader, starting_epoch=0):
    results = []
    best_val_acc = 0.0
    best_epoch = -1
    no_improvement = 0

    for epoch in range(starting_epoch, starting_epoch + num_epochs):

        if epoch >= starting_epoch + 5:
            train_data.transform = get_transforms(resize=224, magnitude=9)
            train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        elif epoch >= starting_epoch + 10:
            train_data.transform =  get_transforms(resize=288, magnitude=12)

        train_loss, train_acc = run_one_epoch(loader=train_loader, model=model, epoch=epoch)
        val_loss, val_acc = validate(loader=val_loader,model=model)

        results.append(
            {
                "epoch": int(epoch),
                "train loss": round(float(train_loss), 4),
                "train acc": round(float(train_acc), 4),
                "val loss": round(float(val_loss), 4),
                "val acc": round(float(val_acc), 4),
            }
        )
        print(f"Epoch ({epoch})- Train Loss: {train_loss}, Train Acc: {train_acc}, Val Loss: {val_loss}, Val Acc: {val_acc}")

        if best_val_acc < val_acc:
            no_improvement = 0
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save({'model_state_dict':model.state_dict(), 'optim_state_dict':optimizer.state_dict(), 'scheduler_state_dict':scheduler.state_dict(), 'epoch':epoch}, ckpt_path + "fold" + str(fold))
        else:
            no_improvement += 1
            if no_improvement > 4:
                print("Early stopping")
                break

        scheduler.step()
        torch.cuda.empty_cache()

    with open(csv_path + "fold" + str(fold) + ".csv", "w") as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'train loss', 'train acc', 'val loss', 'val acc'])
        writer.writeheader()
        writer.writerows(results)

    return best_val_acc

In [11]:
# Fine tune loop
def fine_tuning(model, num_epochs, ckpt_path, csv_path, train_data, train_loader, val_loader, last_acc, starting_epoch=0, learning_rate=0.0001):
    # unfreezing layers for finetuning
    results = []
    best_val_acc = last_acc
    best_epoch = starting_epoch
    no_improvement = 0
    # Unfreeze entire feature extraction
    for param in model.base_model.features.parameters():
        param.require_gradient=True
    # update optimizer with new parameters
    optimizer.add_param_group({'params': base_model.base_model.features.parameters(), 'lr': 0.001})

    # Update transforms for dataset:
    train_data.transform = get_transforms(resize=384, magnitude=14)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    # Unfreeze a few layers at a time
    # frozen_params = list(filter(lambda p: not p.requires_grad, model.parameters()))
    for epoch in range(starting_epoch, starting_epoch + num_epochs):
        # Every epoch, unfreeze a few feature layers

        # for i in range(5):
        #     frozen_params[-1].requires_grad = True
        #     frozen_params.pop(-1)

        # Update optimizer + learning rate
        # unfrozen_params = filter(lambda p: p.requires_grad, model.parameters())
        # learning_rate*=0.1
        # optimizer = optim.Adam(unfrozen_params, lr=scheduler.get_last_lr()) # reduce learning rate every step
        # scheduler.

        train_loss, train_acc = run_one_epoch(loader=train_loader, model=model, epoch=epoch)
        val_loss, val_acc = validate(loader=val_loader,model=model)

        results.append(
            {
                "epoch": int(epoch),
                "train loss": round(float(train_loss), 4),
                "train acc": round(float(train_acc), 4),
                "val loss": round(float(val_loss), 4),
                "val acc": round(float(val_acc), 4),
            }
        )
        print(f"Epoch ({epoch})- Train Loss: {train_loss}, Train Acc: {train_acc}, Val Loss: {val_loss}, Val Acc: {val_acc}")

        if best_val_acc < val_acc:
            no_improvement = 0
            best_val_acc = val_acc
            best_epoch = epoch
            torch.save({'model_state_dict':model.state_dict(), 'epoch':epoch}, ckpt_path + "fold" + str(fold))
        else:
            no_improvement += 1
            if no_improvement > 7:
                print("Early stopping")
                break

        torch.cuda.empty_cache()

    with open(csv_path + "fold" + str(fold) + ".csv", "w") as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'train loss', 'train acc', 'val loss', 'val acc'])
        writer.writeheader()
        writer.writerows(results)
    return best_val_acc


In [12]:
class NumericImageFolder(datasets.ImageFolder):
    """ImageFolder but the labels correspond to the right numbers

    Args:
        root (string): Root directory path.
        transform (callable, optional): A function/transform that  takes in an PIL image
            and returns a transformed version. E.g, ``transforms.RandomCrop``

     Attributes:
        classes (list): List of the class names.
        class_to_idx (dict): Dict with items (class_name, class_index).
        imgs (list): List of (image path, class_index) tuples
    """
    def find_classes(self, directory):
        """
        Overrides the default alphanumeric class-to-index generation.
        """
        # Define your explicit custom mapping here
        custom_mapping = {str(k):k for k in range(0, 100)}

        # Extract the unique class names list
        classes = list(custom_mapping.keys())

        return classes, custom_mapping

In [14]:
# Model Building
class ENetV2STN(nn.Module):
    def __init__(self, num_classes=100, dpr=0.5):
        super(ENetV2STN, self).__init__()

        # Spatial transformer localization-network, from pytorch tutorials
        self.localization = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=7),
            nn.MaxPool2d(2, stride=2),
            nn.ReLU(True),
            nn.Conv2d(8, 10, kernel_size=5),
            nn.MaxPool2d(2, stride=2),
            nn.ReLU(True),
            nn.AdaptiveAvgPool2d((3, 3))    # Fix size for FC localization
        )
        # Regressor for the 3 * 2 affine matrix
        self.fc_loc = nn.Sequential(
            nn.Linear(10 * 3 * 3, 32),
            nn.ReLU(True),
            nn.Linear(32, 3 * 2)
        )
        # Initialize the weights/bias with identity transformation
        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

        # Set up efficientnet base model
        self.base_model = models.efficientnet_v2_s(weights='DEFAULT')
        num_ftrs = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Sequential(
                                        nn.Dropout(p=dpr, inplace=True), # Hyperparameters: p=0.5
                                        nn.Linear(in_features=num_ftrs, out_features=num_classes), # output resized to fit our dataset
                                        )

    # Spatial transformer network forward function
    def stn(self, x):
        xs = self.localization(x)
        xs = xs.view(-1, 10 * 3 * 3)
        theta = self.fc_loc(xs)
        theta = theta.view(-1, 2, 3)

        grid = nn.functional.affine_grid(theta, x.size())
        x = nn.functional.grid_sample(x, grid)

        return x

    def forward(self, x):
            # transform the input
            #x_transform = self.stn(x)
            # feed it into the model
            x = self.base_model(x)

            return x


In [ ]:
class TestSet(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(os.listdir(self.root_dir))

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_id = f"{idx}.jpg"
        img_path = os.path.join(self.root_dir, img_id)
        sample = Image.open(img_path)

        if self.transform:
            sample = self.transform(sample)

        return sample, img_id


In [ ]:
def get_transforms(resize, magnitude, color_jitter_factor=0.2, degrees_rotation=15, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    return tfv2.Compose([
                # Augmentations
                tfv2.RandAugment(num_ops=2, magnitude=magnitude, interpolation=tfv2.InterpolationMode.BILINEAR), # random augmentation
                tfv2.RandomHorizontalFlip(),        # Flip Horizontal
                tfv2.ColorJitter(brightness=color_jitter_factor, contrast=color_jitter_factor), # adjust brightness and contrast

                # Normalization
                tfv2.Resize((resize,resize), interpolation=tfv2.InterpolationMode.BILINEAR),
                tfv2.CenterCrop((resize,resize)),
                tfv2.ToImage(),
                tfv2.ToDtype(torch.float32, scale=True),
                tfv2.Normalize(mean, std)
            ])


In [13]:
# Load Training/Validation Data, transforms, Unlabelled data for pseudolabelling
data_dir = ucsc_cse_144_spring_2026_final_project_path + '/'
BATCH_SIZE = 128
MIX_FACTOR = 0.2
NUM_WORKERS = 0
TRAIN_TEST_RATIO = 0.1
train_transforms = get_transforms(resize=224, magnitude=7)

cutmix = tfv2.CutMix(alpha=MIX_FACTOR, num_classes=100)
mixup = tfv2.MixUp(alpha=MIX_FACTOR, num_classes=100)
cutmix_or_mixup = tfv2.RandomChoice([cutmix, mixup])

train_full = NumericImageFolder(root=data_dir+'train/', transform=train_transforms)
test_set = TestSet(root_dir=data_dir+'test', transform=tfv2.Compose([
    # Normalization only, no augments
    tfv2.Resize((384,384), interpolation=tfv2.InterpolationMode.BILINEAR),
    tfv2.CenterCrop((384,384)),
    tfv2.ToImage(),
    tfv2.ToDtype(torch.float32, scale=True),
    tfv2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
]))


In [16]:
# Model Setup

num_classes= 100
best_val_acc = -1
best_param = 0.0
last_ckpt = ""
ckpt_path = "/content/drive/MyDrive/cse144_final_project_checkpoints/final_06_best_02" # from scratch
csv_path = f"/content/drive/MyDrive/cse144_final_project_checkpoints/csv/final_06_best_02_results"

# Load pre-trained model
base_model = ENetV2STN().to(device)

# refresh parameters list
parameters = list(base_model.parameters())
# freeze layers
for param in parameters:
    param.requires_grad = False

# Unfreeze classifier head
for param in base_model.base_model.classifier.parameters():
    param.requires_grad = True

# # Unfreeze STN layers
# for param in base_model.localization.parameters():
#     param.requires_grad = True

# for param in base_model.fc_loc.parameters():
#     param.requires_grad = True

# Load optimizer, criterion, and scheduler
optimizer = optim.Adam(
    [
        {'params': base_model.base_model.classifier.parameters(), 'lr':0.0005}
    ]
    , lr=0.005)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20) #optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.3)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # cross entropy loss for classification

# Load last saved state
try:
    checkpoint = torch.load(last_ckpt, map_location=device, weights_only=True)
    base_model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optim_state_dict'])
    # Bring to same device
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(device)
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
except:
    print("No previous checkpoint")
    start_epoch = 0


No previous checkpoint


In [1]:
# training
init_state = copy.deepcopy(base_model.state_dict()) # save to reset during folds
num_folds = 1 # training
epochs_per_fold = 25

# For KFOLD Indices
# shuffled_indices = torch.randperm(len(train_data_full)).tolist()
# split_size = len(train_data_full) // num_folds

for fold in range(num_folds):

    # # Split dataset for KFold
    # val_start_idx = fold * split_size
    # val_stop_idx = (fold + 1) * split_size if fold < num_folds else len(train_data)

    # val_idx = shuffled_indices[val_start_idx:val_stop_idx]
    # train_idx = shuffled_indices[:val_start_idx] + shuffled_indices[val_stop_idx:]

    # val_data = Subset(train_data_full, val_idx)
    # train_data = Subset(train_data_full, train_idx)

    # Split dataset for regular training
    train_data, val_data = random_split(train_full, [1-TRAIN_TEST_RATIO, TRAIN_TEST_RATIO])
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print("train/val:", len(train_data), len(val_data))

    # Prepare unlabeled data for pseudolabelling
    unlabeled_data = Subset(test_set, torch.randint(low=0, high=len(test_set), size=(len(test_set)//2)))
    unlabeled_loader = DataLoader(unlabeled_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    print("Training Model")
    best_epoch, last_acc = one_fold(fold=fold, model=base_model, num_epochs=epochs_per_fold, ckpt_path=ckpt_path, csv_path=csv_path, train_data=train_data, train_loader=train_loader, val_loader=val_loader, starting_epoch=start_epoch)
    print(last_acc)

    print("Generating Labels")
    pseudo_labels = generate_labels(loader=unlabeled_loader,model=base_model)

    print("Retraining with Pseudolabels")
    unlabeled_images = torch.stack([unlabeled_data[i][0] for i in range(len(unlabeled_data))])
    pseudo_labeled_data = TensorDataset(unlabeled_images, pseudo_labels)
    combined_data = ConcatDataset([train_data, pseudo_labeled_data])
    combined_loader = DataLoader(combined_data, batch_size=32, shuffle=True)

    best_epoch, last_acc = one_fold(fold=fold, model=base_model, num_epochs=(epochs_per_fold//2), ckpt_path=ckpt_path, csv_path=csv_path, train_data=combined_data, train_loader=combined_loader, val_loader=val_loader, starting_epoch=best_epoch)


    # Get best weights:
    base_model.load_state_dict(torch.load(ckpt_path + "fold" + str(fold))['model_state_dict'])
    base_model.to(device)

    # Split validation data to train for fine tuning (BEST IF REMAINING DATA SIZE >100)
    # remaining_train_data, remaining_val_data = random_split(val_data, [1-train_test_ratio, train_test_ratio])
    # remaining_train_loader = DataLoader(remaining_train_data, batch_size=BATCH_SIZE//2, shuffle=True, num_workers=NUM_WORKERS)
    # remaining_val_loader = DataLoader(remaining_val_data, batch_size=BATCH_SIZE//2, shuffle=False, num_workers=NUM_WORKERS)
    print("Starting Fine Tuning")
    fine_tuning(model=base_model, num_epochs=10, ckpt_path=ckpt_path+"finetuned", csv_path=csv_path+"finetuned", train_data, train_loader=train_loader, val_loader=val_loader, last_acc=last_acc, starting_epoch=best_epoch, learning_rate=0.0001)
    # check check test stats for kfold
    # test_loss, test_acc = validate(test_loader, base_model)

    # Reset state for next fold
    base_model.load_state_dict(copy.deepcopy(init_state))
    base_model.to(device)

SyntaxError: positional argument follows keyword argument (1158709326.py, line 41)

In [ ]:
# Testing, save results as .csv file with imgID | class
import csv
ckpt_path = "/content/drive/MyDrive/cse144_final_project_checkpoints/final_06_best_02finetunedfold0"
data_dir = ucsc_cse_144_spring_2026_final_project_path + '/'

# Load the model
model = ENetV2STN()
model = model.to(device)

state_dict = torch.load(ckpt_path, map_location='cpu')['model_state_dict']
model.load_state_dict(state_dict)
model.eval()

num_workers = 0
test_transforms = tfv2.Compose([
    # Normalization
    tfv2.Resize((480,480), interpolation=tfv2.InterpolationMode.BILINEAR),
    tfv2.CenterCrop((480,480)),
    tfv2.ToImage(),
    tfv2.ToDtype(torch.float32, scale=True),
    tfv2.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_set = TestSet(root_dir=data_dir+'test', transform=test_transforms)
test_loader = DataLoader(test_set, batch_size=1, num_workers=num_workers, shuffle=False)

with open('submission.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ID', 'Label']) # HEADER
    with torch.no_grad():
        for input, id in test_loader:
            input = input.to(device)
            id = id[0]
            output = model(input)
            writer.writerow([id, torch.argmax(output, dim=1).item()])

print("Completed")


Completed
